# Data Preprocessing
This notebook handles the cleaning and preprocessing of the customer data.

In [41]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from datetime import datetime

# Configuration
INPUT_FILE = 'data/customer_info.csv'
OUTPUT_FILE = 'data/customer_info_cleaned.csv'
CURRENT_YEAR = 2025

## 1. Load Data

In [42]:
df = pd.read_csv(INPUT_FILE)
print(f"Data loaded. Shape: {df.shape}")
df.head()

Data loaded. Shape: (16978, 25)


,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
0,11259,Louis Huffman,male,08/22/1948 05:06 AM,0.0,2.0,2.0,3.0,6238.0,818.0,...,1039.0,361.0,457.0,1520.0,210.0,0.623414,2000.0,917650.0,38.683897,-9.172942
1,6538,Msc. Noel Ward,male,02/12/1967 07:51 PM,2.0,1.0,3.0,3.0,4188.0,1048.0,...,1913.0,562.0,67.0,957.0,92.0,0.555637,2017.0,906928.0,38.790490,-9.115064
2,14274,Bsc. Scott Giordano,male,04/10/1986 05:10 PM,0.0,0.0,1.0,1.0,25269.0,4047.0,...,303.0,724.0,442.0,1213.0,85.0,0.259248,2014.0,958107.0,38.775609,-9.112610
3,7796,William Lacy,male,01/03/1999 01:47 AM,0.0,1.0,1.0,3.0,8292.0,1524.0,...,1877.0,560.0,1028.0,1005.0,169.0,0.419730,2024.0,967864.0,38.736641,-9.158548
4,4521,Bsc. Nathan Minor,male,10/21/1957 07:34 AM,3.0,1.0,2.0,3.0,43692.0,3718.0,...,3093.0,1635.0,868.0,745.0,345.0,0.428821,2004.0,971603.0,38.726597,-9.109803


## 2. Categorize Features
Identify categorical and numerical features.

In [ ]:
# Explicit Feature Categorization by Context

categorical_cols = [
    # Demographics
    'customer_name', 'customer_gender', 'customer_birthdate',
    # Behavioral
    'loyalty_card_number'
]

numerical_cols = [
    # Demographics
    'kids_home', 'teens_home', 'latitude', 'longitude',
    
    # Spending
    'lifetime_spend_groceries', 'lifetime_spend_electronics', 
    'lifetime_spend_vegetables', 'lifetime_spend_nonalcohol_drinks',
    'lifetime_spend_alcohol_drinks', 'lifetime_spend_meat',
    'lifetime_spend_fish', 'lifetime_spend_hygiene',
    'lifetime_spend_videogames', 'lifetime_spend_petfood',
    
    # Behavioral
    'number_complaints', 'distinct_stores_visited', 'typical_hour', 
    'lifetime_total_distinct_products', 'percentage_of_products_bought_promotion', 
    'year_first_transaction'
]

print("Categorical Features:", categorical_cols)
print("Numerical Features:", numerical_cols)

## 3. Handle Missing Values

In [44]:
df_clean = df.copy()

# Impute specific columns with Mode
for col in ['kids_home', 'teens_home', 'typical_hour']:
    if col in df_clean.columns:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)

# Impute spending and complaints with 0 (assumption: null = 0)
zero_impute_cols = [
    'number_complaints', 'distinct_stores_visited', 
    'lifetime_spend_groceries', 'lifetime_spend_electronics', 
    'lifetime_spend_vegetables', 'lifetime_spend_nonalcohol_drinks', 
    'lifetime_spend_alcohol_drinks', 'lifetime_spend_meat', 
    'lifetime_spend_fish', 'lifetime_spend_hygiene', 
    'lifetime_spend_videogames', 'lifetime_spend_petfood'
]

for col in zero_impute_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# Check if any missing values remain (except birthdate/loyalty which we handle next)
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

customer_birthdate       68
loyalty_card_number    3143
dtype: int64


## 4. Feature Engineering

In [45]:
df_eng = df_clean.copy()

# 4.1 Age from Birthdate
def calculate_age(birthdate_str):
    if pd.isna(birthdate_str):
        return np.nan
    try:
        dt = datetime.strptime(birthdate_str, '%m/%d/%Y %I:%M %p')
        return CURRENT_YEAR - dt.year
    except:
        return np.nan

if 'customer_birthdate' in df_eng.columns:
    df_eng['Age'] = df_eng['customer_birthdate'].apply(calculate_age)
    df_eng['Age'] = df_eng['Age'].fillna(df_eng['Age'].median())

# 4.2 Loyalty Card Feature
if 'loyalty_card_number' in df_eng.columns:
    df_eng['has_loyalty_card'] = df_eng['loyalty_card_number'].apply(lambda x: 0 if pd.isna(x) else 1)

# 4.3 Drop unused columns
cols_to_drop = ['customer_id', 'customer_name', 'customer_birthdate', 'loyalty_card_number']
df_eng = df_eng.drop(columns=cols_to_drop, errors='ignore')

df_eng.head()

,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,typical_hour,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,...,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,latitude,longitude,Age,has_loyalty_card
0,male,0.0,2.0,2.0,3.0,6238.0,818.0,18.0,1553.0,1479.0,...,361.0,457.0,1520.0,210.0,0.623414,2000.0,38.683897,-9.172942,77.0,1
1,male,2.0,1.0,3.0,3.0,4188.0,1048.0,14.0,2060.0,1549.0,...,562.0,67.0,957.0,92.0,0.555637,2017.0,38.790490,-9.115064,58.0,1
2,male,0.0,0.0,1.0,1.0,25269.0,4047.0,12.0,4905.0,2357.0,...,724.0,442.0,1213.0,85.0,0.259248,2014.0,38.775609,-9.112610,39.0,1
3,male,0.0,1.0,1.0,3.0,8292.0,1524.0,15.0,2281.0,650.0,...,560.0,1028.0,1005.0,169.0,0.419730,2024.0,38.736641,-9.158548,26.0,1
4,male,3.0,1.0,2.0,3.0,43692.0,3718.0,10.0,6068.0,3014.0,...,1635.0,868.0,745.0,345.0,0.428821,2004.0,38.726597,-9.109803,68.0,1


## 5. Encoding and Scaling

In [ ]:
df_proc = df_eng.copy()

# 5.1 One-Hot Encoding
# We check specific categorical columns we know need encoding
if 'customer_gender' in categorical_cols and 'customer_gender' in df_proc.columns:
    df_proc = pd.get_dummies(df_proc, columns=['customer_gender'], drop_first=True)

# 5.2 Scaling
vars_to_scale = []

# Add defined numerical columns
for col in numerical_cols:
    if col in df_proc.columns:
        vars_to_scale.append(col)

# Add engineered numerical feature 'Age'
if 'Age' in df_proc.columns:
    vars_to_scale.append('Age')
    
# Add engineered feature 'has_loyalty_card' (binary, treated as numeric for scaling in this pipeline)
if 'has_loyalty_card' in df_proc.columns:
    vars_to_scale.append('has_loyalty_card')

print(f"Features selected for scaling ({len(vars_to_scale)}):", vars_to_scale)

scaler = StandardScaler()
df_proc[vars_to_scale] = scaler.fit_transform(df_proc[vars_to_scale])

print("Final shape:", df_proc.shape)
df_proc.head()

## 6. Save Output

In [47]:
df_proc.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Saved to data/customer_info_cleaned.csv
